**Part 2 · LangChain** · step 5 of 25  
[← 04 Chat models & providers](04_chat_models_and_providers.ipynb) · [🏠 Roadmap](../README.md) · [06 Prompt templates & chains →](06_prompt_templates_and_chains.ipynb)

# 05 · Messages — the unit of conversation

Messages are the fundamental unit of context for models in LangChain. Each message has:
- **Role** — who is speaking: `system`, `user`/`human`, `assistant`/`ai`, `tool`
- **Content** — text, images, audio, documents…
- **Metadata** — optional: response info, message IDs, token usage

LangChain's message types work the same across every provider.

| Class | Role | Purpose |
|---|---|---|
| `SystemMessage` | system | Sets the model's behaviour / persona |
| `HumanMessage` | user | What the user says |
| `AIMessage` | assistant | What the model replied (may contain `tool_calls`) |
| `ToolMessage` | tool | The result of running a tool, linked by `tool_call_id` |

In [ ]:
from dotenv import load_dotenv

# Reads API keys (OPENAI_API_KEY, GROQ_API_KEY, GOOGLE_API_KEY, TAVILY_API_KEY)
# from the .env file at the AgenticAI root. load_dotenv() walks up parent
# folders until it finds one, so every notebook can share a single .env.
load_dotenv()

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model("groq:qwen/qwen3-32b")

## 1. Text prompts
A plain string is the simplest input — good for one-off tasks where no conversation history is needed.
LangChain wraps it into a single `HumanMessage` for you.

In [ ]:
model.invoke("What is AI?")

## 2. Message prompts
A **list of messages** lets you set a system prompt and pass conversation history.

In [ ]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage("You are a poetry expert"),
    HumanMessage("Write a poem on artificial intelligence"),
]

response = model.invoke(messages)
response.content

In [ ]:
# Token usage lives in the AIMessage metadata
response.usage_metadata

### Passing earlier turns with `AIMessage`
The model has **no memory**. To continue a conversation you resend the earlier turns yourself,
including the model's own previous replies as `AIMessage`s.

In [ ]:
conversation = [
    SystemMessage("You are a helpful assistant."),
    HumanMessage("My name is Ujjawal."),
    AIMessage("Nice to meet you, Ujjawal!"),
    HumanMessage("What is my name?"),
]
model.invoke(conversation).content

### Dictionary format
Equivalent to message objects — handy when the data comes from JSON or an API.

In [ ]:
model.invoke([
    {"role": "system", "content": "You are a helpful coding assistant."},
    {"role": "user", "content": "What is a Python decorator? One line."},
]).content

## 3. Chat models vs. legacy completion models

`langchain_openai` exposes two classes:
- **`ChatOpenAI`** — chat models (`gpt-4o`, `gpt-4.1`, …). Takes messages, returns an `AIMessage`. **Use this.**
- **`OpenAI`** — legacy *text-completion* endpoint. Takes a string, returns a string.
  Chat-only models like `gpt-4o` do **not** work with it.

In [ ]:
from langchain_openai import ChatOpenAI

chat = ChatOpenAI(model="gpt-4o")
chat.invoke([SystemMessage("You are a helpful assistant."), HumanMessage("Hello, how are you?")])

## What does `invoke()` return?

| Object | `invoke()` returns | Contains |
|---|---|---|
| `ChatOpenAI` / `ChatGroq` / `ChatGoogleGenerativeAI` | `AIMessage` | One model response |
| `init_chat_model()` | `AIMessage` | One model response |
| `create_agent()` | `dict` (agent state) | Entire conversation and execution history |

---
✅ **Done with 05 Messages.** Next: [06 Prompt templates & chains →](06_prompt_templates_and_chains.ipynb)  
[🏠 Roadmap](../README.md)